# Module 10: Polynomial Multiplication over GF(2)

**Volume 1 — Foundations, Finite Fields, AES, GCM**  
**Module 10 of 76**

---

## Description

In Module 9 we learned to **add** polynomials over GF(2) using XOR — coefficient addition mod 2 with no carries.  
This module covers **multiplication** of polynomials over GF(2), the operation that underlies the MixColumns step in AES.

The key ideas:
1. **Multiplying terms** uses the exponent rule: $x^i \cdot x^j = x^{i+j}$
2. **Distributing** all terms: every term in $A$ pairs with every term in $B$
3. **Combining like terms mod 2**: any power appearing an even number of times cancels to 0
4. The result is an **unreduced product** — its degree may exceed 7 (byte-size)
5. This feeds directly into **Module 11** (polynomial reduction mod an irreducible polynomial)

---

## Sections

1. Title and Overview *(this cell)*
2. Polynomial Representation and GF(2) Coefficient Arithmetic
3. Implementing Unreduced Polynomial Multiplication
4. Step-by-Step Grid Visualization
5. Worked Example: `0x57 × 0x03`
6. Degree Analysis — Why Products Exceed a Byte
7. Property Verification — Commutativity, Associativity, Distributivity
8. Interactive Explorer
9. Bridge to Reduction — Counting Products That Overflow a Byte
10. Summary and Bridge to Module 11

---
## Section 2: Polynomial Representation and GF(2) Coefficient Arithmetic

### Representation

A polynomial over GF(2) has coefficients in $\{0, 1\}$.  
We represent it as a **`frozenset` of exponents** — only the exponents with coefficient 1 are stored.

For example, $x^6 + x^4 + x^2 + x + 1$ corresponds to `frozenset({6, 4, 2, 1, 0})`.

A byte value like `0x57 = 0b01010111` encodes the polynomial via its bit positions:
- Bit 6 → $x^6$, Bit 4 → $x^4$, Bit 2 → $x^2$, Bit 1 → $x$, Bit 0 → $1$

### GF(2) Coefficient Multiplication Truth Table

| $a$ | $b$ | $a \cdot b$ |
|-----|-----|-------------|
|  0  |  0  |      0      |
|  0  |  1  |      0      |
|  1  |  0  |      0      |
|  1  |  1  |      1      |

Coefficient multiplication is ordinary integer multiplication (AND). Only $1 \cdot 1 = 1$ produces a non-zero term.

In [ ]:
# ── Section 2: Polynomial Representation ────────────────────────────────────

def poly_from_int(n):
    """
    Convert an integer to a frozenset of exponents over GF(2).
    Each set bit at position i contributes the term x^i.

    Examples:
        poly_from_int(0x57) == frozenset({6, 4, 2, 1, 0})
        poly_from_int(0x03) == frozenset({1, 0})
        poly_from_int(0)    == frozenset()
    """
    return frozenset(i for i in range(n.bit_length()) if (n >> i) & 1)


def poly_to_str(p):
    """
    Format a frozenset of exponents as a human-readable polynomial string.
    Terms are sorted in descending degree order.

    Examples:
        poly_to_str(frozenset({6,4,2,1,0})) == 'x^6 + x^4 + x^2 + x + 1'
        poly_to_str(frozenset())             == '0'
    """
    if not p:
        return '0'
    terms = []
    for exp in sorted(p, reverse=True):
        if exp == 0:
            terms.append('1')
        elif exp == 1:
            terms.append('x')
        else:
            terms.append(f'x^{exp}')
    return ' + '.join(terms)


# ── Demonstration ────────────────────────────────────────────────────────────

examples = [
    (0x57, 'AES S-box example'),
    (0x03, 'Simple two-term poly'),
    (0xFF, 'All 8 bits set'),
    (0x01, 'Constant polynomial 1'),
    (0x00, 'Zero polynomial'),
]

print(f"{'Hex':>6}  {'Binary':>10}  {'Frozenset (exponents)':>26}  Polynomial string")
print('-' * 75)
for val, label in examples:
    p = poly_from_int(val)
    print(f"  {val:#04x}  {bin(val):>10}  {str(sorted(p, reverse=True)):>26}  {poly_to_str(p)}")
    print(f"        ({label})")

print()
print('GF(2) coefficient multiplication truth table:')
print(f"  {'a':>3}  {'b':>3}  {'a·b':>5}")
for a in (0, 1):
    for b in (0, 1):
        print(f"  {a:>3}  {b:>3}  {a*b:>5}")

---
## Section 3: Implementing Unreduced Polynomial Multiplication

### The Algorithm

Multiplying $A(x) \cdot B(x)$ over GF(2) proceeds in three steps:

**Step 1 — Distribute (partial products)**  
For each exponent $e_a \in A$ and each exponent $e_b \in B$, produce the term $x^{e_a + e_b}$.  
This gives $|A| \times |B|$ partial product terms (with possible repetition).

**Step 2 — Collect and count**  
Tally how many times each exponent appears among the partial products.

**Step 3 — Reduce coefficients mod 2 (XOR-cancel)**  
Keep only exponents that appear an **odd** number of times.  
Any exponent appearing an even number of times has coefficient $0 + 0 = 0$ (mod 2) and disappears.

### Worked Example

$(x + 1)(x^2 + x + 1)$:

| $e_a$ | $e_b$ | $e_a + e_b$ | term |
|--------|--------|-------------|------|
| 1 | 2 | 3 | $x^3$ |
| 1 | 1 | 2 | $x^2$ |
| 1 | 0 | 1 | $x$   |
| 0 | 2 | 2 | $x^2$ |
| 0 | 1 | 1 | $x$   |
| 0 | 0 | 0 | $1$   |

Counts: $x^3$ → 1, $x^2$ → 2 (cancels!), $x$ → 2 (cancels!), $1$ → 1  
Result: $x^3 + 1$ ✓

In [ ]:
# ── Section 3: Unreduced Polynomial Multiplication ──────────────────────────

def poly_mul_unreduced(a, b):
    """
    Multiply two polynomials over GF(2) without reducing modulo an
    irreducible polynomial. Returns a sorted list of surviving exponents
    (those appearing an odd number of times in the distribution).

    Parameters
    ----------
    a, b : frozenset of int  — exponents with coefficient 1

    Returns
    -------
    list of int (sorted descending) — exponents of the product
    """
    # Step 1 & 2: collect all partial products, count occurrences
    counts = {}  # exponent -> count
    for ea in a:
        for eb in b:
            exp = ea + eb
            counts[exp] = counts.get(exp, 0) + 1

    # Step 3: keep only odd-count exponents (coefficient 1 mod 2)
    survivors = sorted(
        (exp for exp, cnt in counts.items() if cnt % 2 == 1),
        reverse=True
    )
    return survivors


def poly_mul_to_frozenset(a, b):
    """Convenience wrapper returning a frozenset instead of a list."""
    return frozenset(poly_mul_unreduced(a, b))


# ── Verification: (x+1)(x^2+x+1) = x^3+1 ────────────────────────────────────

a = poly_from_int(0b011)   # x + 1
b = poly_from_int(0b111)   # x^2 + x + 1
product = poly_mul_unreduced(a, b)

print('Worked example: (x+1)(x^2+x+1)')
print(f'  A = {poly_to_str(a)}')
print(f'  B = {poly_to_str(b)}')
print()
print('  Partial products (all ea+eb pairs):')
from collections import Counter
counts = Counter()
for ea in sorted(a, reverse=True):
    for eb in sorted(b, reverse=True):
        exp = ea + eb
        counts[exp] += 1
        print(f'    x^{ea} * x^{eb}  -->  x^{exp}')

print()
print('  Counts per exponent:')
for exp in sorted(counts, reverse=True):
    status = 'SURVIVES' if counts[exp] % 2 == 1 else 'CANCELS (even count)'
    print(f'    x^{exp}: appears {counts[exp]} time(s) -- {status}')

print()
print(f'  Product = {poly_to_str(frozenset(product))}')
assert frozenset(product) == frozenset({3, 0}), 'Expected x^3 + 1'
print('  Verification: x^3 + 1  ✓')

---
## Section 4: Step-by-Step Grid Visualization

A multiplication grid (analogous to a times-table grid for integers) shows every partial product $x^{e_a} \cdot x^{e_b} = x^{e_a + e_b}$.  
Terms that cancel (appear an even number of times) are marked with `(X)`.  
Surviving terms are marked with `(*)` and collected into the final sum.

In [ ]:
# ── Section 4: Grid Visualization ───────────────────────────────────────────

def mul_grid(a, b, label_a='A', label_b='B'):
    """
    Print a text grid showing each partial product for A*B over GF(2).
    Mark terms that cancel (even count) vs terms that survive (odd count).
    """
    a_exps = sorted(a, reverse=True)
    b_exps = sorted(b, reverse=True)

    # Count all occurrences
    counts = Counter()
    for ea in a_exps:
        for eb in b_exps:
            counts[ea + eb] += 1

    print(f'  Multiplication grid: ({poly_to_str(a)}) × ({poly_to_str(b)})')
    print()

    # Header row
    col_w = 8
    header = ' ' * 10 + ''.join(f'{"x^"+str(eb) if eb>1 else ("x" if eb==1 else "1"):^{col_w}}' for eb in b_exps)
    print(header)
    print(' ' * 10 + '-' * (col_w * len(b_exps)))

    # Rows
    for ea in a_exps:
        row_label = f'  {"x^"+str(ea) if ea>1 else ("x" if ea==1 else "1"):>6} |'
        cells = []
        for eb in b_exps:
            exp = ea + eb
            tag = '*' if counts[exp] % 2 == 1 else 'X'
            term = f'x^{exp}' if exp > 1 else ('x' if exp == 1 else '1')
            cells.append(f'{term}({tag})'.center(col_w))
        print(row_label + ''.join(cells))

    print()
    print('  Legend:  (*) = term survives (appears odd # of times)')
    print('           (X) = term cancels  (appears even # of times, coefficient 0 mod 2)')
    print()

    # Collect survivors
    survivors = sorted((e for e, c in counts.items() if c % 2 == 1), reverse=True)
    result_set = frozenset(survivors)

    print(f'  All partial product exponents (with counts):')
    for exp in sorted(counts, reverse=True):
        c = counts[exp]
        status = 'survives' if c % 2 == 1 else 'cancels'
        term = f'x^{exp}' if exp > 1 else ('x' if exp == 1 else '1')
        print(f'    {term:>6}: count={c}  -> {status}')

    print()
    print(f'  Result = {poly_to_str(result_set)}')
    return result_set


# ── Example: (x^2+x+1) × (x^3+x) ───────────────────────────────────────────

a4 = poly_from_int(0b0111)   # x^2 + x + 1
b4 = poly_from_int(0b1010)   # x^3 + x

result4 = mul_grid(a4, b4)

---
## Section 5: Worked Example — `0x57 × 0x03`

This is the canonical AES MixColumns example.  
`0x57` = $x^6 + x^4 + x^2 + x + 1$ and `0x03` = $x + 1$.  

We expect the unreduced product to be $x^7 + x^6 + x^5 + x^4 + x^3 + 1$ (degree 7, just fits in a byte — the reduction step happens in Module 11 for larger products).

In [ ]:
# ── Section 5: 0x57 × 0x03 Step-by-Step ─────────────────────────────────────

a5 = poly_from_int(0x57)   # x^6 + x^4 + x^2 + x + 1
b5 = poly_from_int(0x03)   # x + 1

print('═' * 60)
print('  Worked Example: 0x57 × 0x03 in GF(2)[x]')
print('═' * 60)
print()
print(f'  0x57 = {bin(0x57)} = {poly_to_str(a5)}')
print(f'  0x03 = {bin(0x03)} = {poly_to_str(b5)}')
print()

# Show all partial products
print('  Step 1 — All partial products (ea + eb):')
counts5 = Counter()
for ea in sorted(a5, reverse=True):
    for eb in sorted(b5, reverse=True):
        exp = ea + eb
        counts5[exp] += 1
        ea_str = f'x^{ea}' if ea > 1 else ('x' if ea == 1 else '1')
        eb_str = f'x^{eb}' if eb > 1 else ('x' if eb == 1 else '1')
        res_str = f'x^{exp}' if exp > 1 else ('x' if exp == 1 else '1')
        print(f'    {ea_str:>4} · {eb_str:<4} = {res_str}')

print()
print('  Step 2 — Counts and cancellations:')
for exp in sorted(counts5, reverse=True):
    c = counts5[exp]
    term = f'x^{exp}' if exp > 1 else ('x' if exp == 1 else '1')
    parity = 'ODD  -> survives' if c % 2 == 1 else 'EVEN -> cancels'
    print(f'    {term:>4}: count={c}  ({parity})')

print()
product5 = poly_mul_unreduced(a5, b5)
result5 = frozenset(product5)
print(f'  Step 3 — Final unreduced product:')
print(f'    0x57 × 0x03 = {poly_to_str(result5)}')
print()

expected5 = frozenset({7, 6, 5, 4, 3, 0})
assert result5 == expected5, f'Expected {poly_to_str(expected5)}, got {poly_to_str(result5)}'
print('  Verification: x^7 + x^6 + x^5 + x^4 + x^3 + 1  ✓')
print()
print('  Note: degree = 7, which just fits in a byte (degree < 8).')
print('  In Module 11 we will reduce products of degree >= 8.')

---
## Section 6: Degree Analysis — Why Products Exceed a Byte

### Degrees and Byte Sizes

A **byte** (values 0–255) corresponds to polynomials of degree **at most 7** (8 bits, bit positions 0–7).  

- **Addition** (XOR) can never increase the degree: $\deg(A + B) \leq \max(\deg A, \deg B)$
- **Multiplication** adds degrees: $\deg(A \cdot B) = \deg A + \deg B$

The worst case is `0xFF × 0xFF`:  
$\deg(0xFF) = 7$ (bits 0–7 all set), so the unreduced product can reach degree $7 + 7 = 14$.  
A degree-14 polynomial needs **15 coefficients** — far beyond a single byte.

This is why we need **reduction modulo an irreducible polynomial** (Module 11) to bring the result back into the byte-sized GF(2⁸).

In [ ]:
# ── Section 6: Degree Analysis ───────────────────────────────────────────────

def poly_degree(p):
    """
    Return the degree of a polynomial represented as a frozenset (or list)
    of exponents. The zero polynomial returns -1 by convention.
    """
    if not p:
        return -1
    return max(p)


print('Degree of selected polynomials:')
for val in [0x00, 0x01, 0x02, 0x57, 0xFF]:
    p = poly_from_int(val)
    print(f'  {val:#04x} -> deg={poly_degree(p):>2}  ({poly_to_str(p)})')

print()

# Maximum degrees for addition vs multiplication
print('Addition vs multiplication — max degree comparison:')
p_ff = poly_from_int(0xFF)
add_result = frozenset(p_ff.symmetric_difference(p_ff))   # XOR = 0
mul_result = frozenset(poly_mul_unreduced(p_ff, p_ff))

print(f'  0xFF + 0xFF (XOR) = {poly_to_str(add_result)}  (degree {poly_degree(add_result)})')
print(f'  0xFF * 0xFF       = {poly_to_str(mul_result)}')
print(f'    degree = {poly_degree(mul_result)}  (needs {poly_degree(mul_result)+1} bits to represent)')
print()

# Show a range of products and their degrees
print('Degree of unreduced product for several byte pairs:')
pairs = [
    (0x01, 0xFF, 'identity times max'),
    (0x02, 0x80, 'x times x^7 = x^8 (needs reduction)'),
    (0x57, 0x03, 'AES example (degree 7 — no reduction needed)'),
    (0x57, 0x13, 'AES MixColumns full'),
    (0xFF, 0xFF, 'worst case'),
]
print(f'  {"A":>6}  {"B":>6}  {"deg(A*B)":>10}  Note')
print('  ' + '-' * 60)
for va, vb, note in pairs:
    pa = poly_from_int(va)
    pb = poly_from_int(vb)
    prod = poly_mul_unreduced(pa, pb)
    deg = poly_degree(prod) if prod else -1
    overflow = '  <-- exceeds byte!' if deg > 7 else ''
    print(f'  {va:#06x}  {vb:#06x}  {deg:>10}  {note}{overflow}')

print()
print('Rule: deg(A*B) = deg(A) + deg(B)  [over GF(2), for non-zero polynomials]')
print('      Addition never increases degree; multiplication always adds degrees.')

---
## Section 7: Property Verification

Polynomial multiplication over GF(2) satisfies the standard ring axioms.  
We verify three properties over a sample of byte values:

1. **Commutativity**: $A \cdot B = B \cdot A$ for all $A, B$
2. **Associativity**: $(A \cdot B) \cdot C = A \cdot (B \cdot C)$ for all $A, B, C$
3. **Distributivity over addition**: $A \cdot (B + C) = (A \cdot B) + (A \cdot C)$

Here `+` is XOR (polynomial addition from Module 9), which is the `symmetric_difference` of exponent sets.

In [ ]:
# ── Section 7: Property Verification ────────────────────────────────────────

def poly_add(a, b):
    """
    Add two GF(2) polynomials (= XOR of their exponent sets).
    Returns a frozenset.
    """
    return frozenset(a.symmetric_difference(b))


# Sample of byte values to test
SAMPLE = [0x00, 0x01, 0x02, 0x03, 0x57, 0x83, 0xAB, 0xFF]
polys  = {v: poly_from_int(v) for v in SAMPLE}

comm_fails  = 0
assoc_fails = 0
dist_fails  = 0

# Commutativity: a*b == b*a
for va in SAMPLE:
    for vb in SAMPLE:
        ab = frozenset(poly_mul_unreduced(polys[va], polys[vb]))
        ba = frozenset(poly_mul_unreduced(polys[vb], polys[va]))
        if ab != ba:
            comm_fails += 1

# Associativity: (a*b)*c == a*(b*c)
for va in SAMPLE:
    for vb in SAMPLE:
        ab = frozenset(poly_mul_unreduced(polys[va], polys[vb]))
        for vc in SAMPLE:
            lhs = frozenset(poly_mul_unreduced(ab, polys[vc]))
            bc  = frozenset(poly_mul_unreduced(polys[vb], polys[vc]))
            rhs = frozenset(poly_mul_unreduced(polys[va], bc))
            if lhs != rhs:
                assoc_fails += 1

# Distributivity: a*(b+c) == (a*b) + (a*c)
for va in SAMPLE:
    for vb in SAMPLE:
        for vc in SAMPLE:
            bpc  = poly_add(polys[vb], polys[vc])
            lhs  = frozenset(poly_mul_unreduced(polys[va], bpc))
            ab   = frozenset(poly_mul_unreduced(polys[va], polys[vb]))
            ac   = frozenset(poly_mul_unreduced(polys[va], polys[vc]))
            rhs  = poly_add(ab, ac)
            if lhs != rhs:
                dist_fails += 1

n2 = len(SAMPLE) ** 2
n3 = len(SAMPLE) ** 3

print('Property Verification over GF(2)[x]')
print(f'  Sample values: {[hex(v) for v in SAMPLE]}')
print()
print(f'  Commutativity   a*b == b*a       {n2:>5} pairs tested:   {comm_fails} failures  ', end='')
print('✓' if comm_fails == 0 else '✗ FAILED')

print(f'  Associativity   (a*b)*c == a*(b*c) {n3:>5} triples tested: {assoc_fails} failures  ', end='')
print('✓' if assoc_fails == 0 else '✗ FAILED')

print(f'  Distributivity  a*(b+c)==a*b+a*c  {n3:>5} triples tested: {dist_fails} failures  ', end='')
print('✓' if dist_fails == 0 else '✗ FAILED')

print()
if comm_fails == assoc_fails == dist_fails == 0:
    print('  All ring axioms verified for the sample set.  ✓')
else:
    print('  Some properties FAILED — check implementation.')

print()
print('  Quick spot-check examples:')
for va, vb in [(0x57, 0x03), (0x83, 0x02), (0xAB, 0xFF)]:
    ab = frozenset(poly_mul_unreduced(polys[va], polys[vb]))
    ba = frozenset(poly_mul_unreduced(polys[vb], polys[va]))
    print(f'    {va:#04x} * {vb:#04x} == {vb:#04x} * {va:#04x} ? {"yes" if ab == ba else "no"}')

---
## Section 8: Interactive Explorer

Use `mul_explorer(hex_a, hex_b)` to walk through any two byte-valued polynomials step by step.  
Pass hex strings such as `'0x57'` and `'0x03'`.

Several preset examples are called at the bottom of the cell.

In [ ]:
# ── Section 8: Interactive Explorer ─────────────────────────────────────────

def mul_explorer(hex_a, hex_b):
    """
    Show a complete step-by-step polynomial multiplication over GF(2).

    Parameters
    ----------
    hex_a, hex_b : str
        Hex strings such as '0x57' or '0x03'. Byte values 0x00–0xFF.

    Returns
    -------
    frozenset : exponents of the unreduced product
    """
    va = int(hex_a, 16)
    vb = int(hex_b, 16)

    if not (0 <= va <= 0xFF and 0 <= vb <= 0xFF):
        raise ValueError(f'Values must be in 0x00–0xFF, got {hex_a}, {hex_b}')

    a = poly_from_int(va)
    b = poly_from_int(vb)

    sep = '─' * 62
    print(sep)
    print(f'  Polynomial Multiplication: {hex_a} × {hex_b}')
    print(sep)
    print(f'  {hex_a} = {bin(va):>10}  ->  {poly_to_str(a)}')
    print(f'  {hex_b} = {bin(vb):>10}  ->  {poly_to_str(b)}')
    print()

    if not a or not b:
        print('  One operand is the zero polynomial; product = 0')
        print(sep)
        return frozenset()

    # Partial products
    print('  Partial products:')
    counts = Counter()
    for ea in sorted(a, reverse=True):
        ea_str = f'x^{ea}' if ea > 1 else ('x' if ea == 1 else '1')
        for eb in sorted(b, reverse=True):
            exp = ea + eb
            counts[exp] += 1
            eb_str = f'x^{eb}' if eb > 1 else ('x' if eb == 1 else '1')
            res_str = f'x^{exp}' if exp > 1 else ('x' if exp == 1 else '1')
            print(f'    {ea_str:>5} · {eb_str:<5}  =  {res_str}')

    print()
    print('  Coefficient counts (mod 2 reduction):')
    survivors = []
    for exp in sorted(counts, reverse=True):
        c = counts[exp]
        term = f'x^{exp}' if exp > 1 else ('x' if exp == 1 else '1')
        coeff = c % 2
        fate = f'coefficient = {coeff}  ->  {"KEEP" if coeff else "DROP"}'
        print(f'    {term:>6}: appears {c}×  {fate}')
        if coeff:
            survivors.append(exp)

    result = frozenset(survivors)
    deg = poly_degree(result) if result else -1

    print()
    print(f'  Unreduced product = {poly_to_str(result)}')
    print(f'  Degree = {deg}  ({"fits in byte (degree <= 7)" if deg <= 7 else "EXCEEDS byte (degree > 7) -- reduction required"})')
    print(sep)
    return result


# ── Preset examples ───────────────────────────────────────────────────────────

_ = mul_explorer('0x57', '0x03')   # AES MixColumns classic
print()
_ = mul_explorer('0x02', '0x80')   # x · x^7 = x^8 -- requires reduction
print()
_ = mul_explorer('0x01', '0xAB')   # identity
print()
_ = mul_explorer('0x00', '0xFF')   # zero operand

---
## Section 9: Bridge to Reduction — Counting Products That Overflow a Byte

How many of the $256 \times 256 = 65536$ byte-pair products result in a polynomial of degree $\geq 8$?  
Those products cannot be represented as a single byte and require reduction modulo the AES irreducible polynomial $m(x) = x^8 + x^4 + x^3 + x + 1$ (Module 11).

In [ ]:
# ── Section 9: Bridge to Reduction ──────────────────────────────────────────

print('Counting byte-pair products that exceed degree 7...')
print('(This may take a few seconds — 65 536 multiplications)')
print()

overflow_count = 0
overflow_examples = []   # store a few examples for display

for va in range(256):
    pa = poly_from_int(va)
    for vb in range(256):
        pb = poly_from_int(vb)
        prod = poly_mul_unreduced(pa, pb)
        if prod and max(prod) >= 8:
            overflow_count += 1
            if len(overflow_examples) < 10 and va <= vb:  # symmetric, collect upper triangle
                overflow_examples.append((va, vb, frozenset(prod)))

total = 256 * 256
fits  = total - overflow_count

print(f'  Total byte-pair combinations:       {total:>6}')
print(f'  Products with degree <= 7 (fit):    {fits:>6}  ({100*fits/total:.1f}%)')
print(f'  Products with degree >= 8 (overflow): {overflow_count:>6}  ({100*overflow_count/total:.1f}%)')
print()
print('  The vast majority of byte multiplications overflow — this is why')
print('  GF(2^8) arithmetic ALWAYS applies modular reduction after multiplication.')
print()

print('  Sample overflow products:')
print(f'  {"A":>6}  {"B":>6}  {"deg":>5}  Product polynomial')
print('  ' + '-' * 65)
for va, vb, prod in overflow_examples:
    deg = poly_degree(prod)
    print(f'  {va:#06x}  {vb:#06x}  {deg:>5}  {poly_to_str(prod)}')

print()
print('  These products are the inputs to the reduction step in Module 11.')
print('  The AES irreducible polynomial m(x) = x^8 + x^4 + x^3 + x + 1')
print('  (0x11B) is used to reduce them back into a byte-sized result.')

---
## Section 10: Summary and Bridge to Module 11

### Key Rules for Polynomial Multiplication over GF(2)

| Rule | Formula | Notes |
|------|---------|-------|
| Term multiplication | $x^i \cdot x^j = x^{i+j}$ | Exponents add |
| Distribution | $A \cdot B = \sum_{e_a \in A,\ e_b \in B} x^{e_a+e_b}$ | All pairs of terms |
| Coefficient arithmetic | $c_k = \sum_{{i+j=k}} a_i b_j \pmod{2}$ | XOR cancels duplicates |
| Degree | $\deg(A \cdot B) = \deg A + \deg B$ | Degrees add |
| Commutativity | $A \cdot B = B \cdot A$ | Order doesn't matter |
| Associativity | $(A \cdot B) \cdot C = A \cdot (B \cdot C)$ | Grouping doesn't matter |
| Distributivity | $A \cdot (B + C) = A \cdot B + A \cdot C$ | $+$ is XOR here |
| Identity | $A \cdot 1 = A$ | Multiplying by 1 is a no-op |
| Zero | $A \cdot 0 = 0$ | Zero absorbs |

### Why Reduction Is Needed

Over GF(2)[x] the product of two degree-$\leq 7$ polynomials can reach degree 14.  
To work in **GF(2⁸)** — the 256-element field used in AES — we must reduce every product **modulo an irreducible polynomial of degree 8**.

### Bridge to Module 11

**Module 11: Polynomial Reduction and Irreducible Polynomials** covers:

- What makes a polynomial **irreducible** (cannot be factored) over GF(2)
- **Polynomial division** over GF(2): computing remainder $A \mod m(x)$
- The AES choice: $m(x) = x^8 + x^4 + x^3 + x + 1$ (hex `0x11B`)
- Why any irreducible degree-8 polynomial defines a valid GF(2⁸)
- Combining Modules 10 and 11 to implement complete GF(2⁸) multiplication

---

> **Recap:** This module showed that polynomial multiplication over GF(2) is mechanical (distribute, then XOR-cancel duplicates) and algebraically well-behaved (commutative, associative, distributive).  
> The only obstacle to its use in AES is that products can exceed byte size — a problem solved cleanly by modular reduction in Module 11.